In [2]:
import secrets
from ipaddress import ip_address
from operator import truediv
from wsgiref import util

from requests.compat import numeric_types

'''

Scenario :
AIS monitors traffic coming in and out of the local network at an IP level and identifies malformed packages

How the AIS works :
Lymphocytes detect patterns not in the self set (= detector set = unusual packages)
When enough of the lymphocytes detectors detect elements of the detector set
Then the lymphocytes activation threshold is reached which triggers an alarm

Plan for gerüst :
1. write lymphocyte class
    1a. create antibody class
        a.set up detects method specifics of detects
            a1. Implement detects based on packages in scapy this is implemented when detects is true when antibody = package
                - decide on maths
                    - determine level of threshhold for detection = stimlation = how similar does antibody have to be to package to trigger alarm
                    - decide what datastructures are we using and how is the comparison performed
                    - in the first version use maths from demo (if it can be used on larger packages)
2. create self set based on normal packages in scapy
    a. create example packets to test that the normal_traffic and antibody connect correctly
    b. download a file (write down details of training data to account for biases later) with example packets (for malicious and not malicious) and use them as example traffic (download, filter for http/https, convert to bits)
3. write the self set (lyphocytes that dont detect normal_packets) using negative selection

Future Improvements :
-> create a virtual environment to send the packets and lympocytes through using mininet so that lymphocites are trained in real time
    -> change the packages/traffic based on the latest threats?
-> add lymphocites dying like the architecture.. paper and being replaced - a population of lyphocytes
-> add memory detectors (and limit them) like the architecture .. paper
-> model primary response (lymphocytes that have detected a new pathogen multiply because the others dont have the antibodies to recognise it)  and secondary response based on this paper https://www.researchgate.net/profile/Steven-Hofmeyr/publication/12197865_Architecture_for_an_Artificial_Immune_System/links/00b7d538f8550ca681000000/Architecture-for-an-Artificial-Immune-System.pdf (generally do it like that paper to improve)
-> make it so sensitivity level mimics cytokines (signaling other lymphocytes) if there were already activations in the region?
-> Have ABNORMAL_HTTP_TRAFFIC simulate not just malformed packages but also malicious packages that follow the rules of the protocol packets but simulate : IPV4 packet fragmentation attacks, TCP Null scan, TCP FIN scan, TCP XMas Scan, TCP flag manipulation, TCP sequence number anomalies, ICMP based attacks, DNS tunneling, DNS poisoning, DHCP spoofing, protocol fuzzing
-> define the threshold of activation not for once lyphocyte but as a group so that - for example - it the AIS is deployed across several vlans of a company network their joint activation thresshold enables us to recognise attack patterns across several
vlans (that use different firwalls and siems so that the detection of overall patterns is an isusue).
-> instead of making the generated antibodies random make them more like possible malicious packages without narrowing down the detector coverage

Distant Future Improvements
-> work out your own problem representation for how antigens are constructed (see review paper https://staff.fmi.uvt.ro/~daniela.zaharie/am2016/proiecte/tehnici/AIS/AIS_advances.pdf)
-> use flow records instead of raw packages (so the relationship between the packages can be considered)
-> inspect packets at TCP and DNS level
'''

'\nBasic Premise\nLymphocytes detect patterns not in the self set (= detector set = unusual packages)\nWhen enough of the lymphocytes detectors detect elements of the detector set\nThen the lymphocytes activation threshold is reached which triggers an alarm\n\nPlan for gerüst\n1. write lymphocyte class\n    1a. create antibody class\n        a.set up detects method specifics of detects left vague because we dont know how to process the packages provided by this library effiently\n2. create self set based on normal packages in scapy\n3. write the lyphocyte set (lyphocytes that dont detect packages in the self set) using negative selection\n'

In [7]:
import random
import warnings
from scapy.layers.inet import IP, TCP
from scapy.packet import Packet
from enum import Enum, auto
import ipaddress
from bitarray import bitarray as BitArray
"""use this to use packages from the huge PCAP file whithout having to download it all"""
"""save packets from stream to use as mixed input then work out if detection was correct
based on csv files"""
from scapy.utils import PcapReader
import numpy as np
import pytest


num_packets_in_collection = 2 #rows
num_bits_in_packet = 80 #columns
Normal_Packet_Collection = np.empty((num_packets_in_collection, num_bits_in_packet), dtype = np.bool_)
Abnormal_Packet_Collection = np.empty((num_packets_in_collection, num_bits_in_packet), dtype = np.bool_)
Mixed_Packet_Collection = np.empty((num_packets_in_collection, num_bits_in_packet), dtype = np.bool_)

"used to identify duplicate lymphocytes"
Lymphocyte_Set = None


"Defines the type of traffic"
class PacketCollectionType(Enum):
        NORMAL = auto()
        ABNORMAL = auto()
        MIXED = auto()

packet_collections = {
    PacketCollectionType.NORMAL : Normal_Packet_Collection,
    PacketCollectionType.ABNORMAL : Abnormal_Packet_Collection,
    PacketCollectionType.MIXED : Mixed_Packet_Collection
}


"""This function creates a list representing the self set. Any detector the detect method determines too close to an element of the self set is deleted"""
def create_packet_set(packet_collection : PacketCollectionType,src_IP, dst_IP,dport):
        #TODO: call a method that creates the packages and save them as a WHAT?
        #only save/use (source host IP address,destination host IP address,TCP service/port number) LISYS
        """"paper calss these a datapath triple : so it describes a kind of connection """
        if packet_collection == PacketCollectionType.NORMAL:
            self_set : list[Packet] = []
            while len(self_set) < num_packets_in_collection:
                packet = IP(src=random.choice(src_IP),dst = random.choice(dst_IP))/TCP(sport = random.choice(dport))
                self_set.append(packet)
                #check that self set is a list of valid packages
                #convert self_set to binary
            return self_set
        elif packet_collection == PacketCollectionType.ABNORMAL:
            pass
        elif packet_collection == PacketCollectionType.MIXED:
            pass
#TODO the list[list] is a list of lists packets each item is a feild. turn into numpy array in final version
#TODO improve codes by giving it to AI and asking what level of experience the code suggests then change the things that mark it as beginner code
"""filters out unsuable packets (IPv6 or not first in the connect) and converts the usable packets to numpy bit arrays which are added to 2D numbpy bit arrays that are either Normal, Abnormal or Mixed"""
#TO DO : should return np.ndarray but for now its none
def convert_collection_to_bits(packet_collection : list[list]) -> np.ndarray:
    list_number = 0
    element_number = 0
    packet_collection_in_bits = []
    packet_in_bits= []
    #step 1 turn each value into a 2D numpy array
    #length of packet_collection is the number of packets (so lists) in the list
    #TODO : turn this into a for loop
    while list_number < len(packet_collection):
        #not using np.array immediately to be more computationally efficient
       # if not is_packet_usable(packet_collections[list_number]):
          #  ++element_number
        #sport is allways the third field ensuring order stays consistent
        if type(packet_collection[list_number][element_number]) == int :
            if element_number == 2:
                sport_bits = BitArray(format(packet_collection[list_number][element_number],"016b"))
                packet_in_bits.extend(sport_bits)
                if list_number == 0 and element_number == 2:
                    print(("sport ip is",sport_bits, "in bits"))
                element_number += 1
                #if we have reached the third element we are at the end of the packet/list
                #so we go to a new list
                list_number += 1
                # and we reset element number for the next list
                element_number = 0
                #turn finished packet into np array
                packet_in_bits_array = np.array(packet_in_bits)
                #clear packet for next iteration
                packet_in_bits = []
                #add np array containing binary packet to the collection
                packet_collection_in_bits.append(packet_in_bits_array)
            else:
                warnings.warn("third element of tuple should contain an int and should contain sport currently third element contains" + str(packet_collection[list_number][element_number]))
        #if the field is an ip address
        elif type(packet_collection[list_number][element_number]) == str :
            if element_number == 0:
                src_ip_bytes = ipaddress.IPv4Address(packet_collection[list_number][element_number]).packed
                src_ip_bits = BitArray()
                src_ip_bits.frombytes(src_ip_bytes)
                packet_in_bits.extend(src_ip_bits)
                if list_number == 0 and element_number == 0 :
                    print(("src ip is ", src_ip_bits,"in bits"))
                element_number +=1
            if element_number == 1:
                dst_ip_bytes = ipaddress.IPv4Address(packet_collection[list_number][element_number]).packed
                dst_ip_bits = BitArray()
                dst_ip_bits.frombytes(dst_ip_bytes)
                packet_in_bits.extend(dst_ip_bits)
                if list_number == 0 and element_number == 1 :
                    print(("dst ip is",dst_ip_bits, "in bits"))
                element_number += 1
    packet_collection_in_bits_array = np.array(packet_collection_in_bits)
    check_bit_conversion(packet_collection_in_bits_array,packet_collection)
    return packet_collection_in_bits_array



    #step 2 after conversion each array is added to a np matrix to create one packet
    #step 3 the packet is added to the 2D numpy array
#TODO write an indepentend converter with different logic to have values to compare the packet to
"""sampling random packets for errors in the conversion function"""
def check_bit_conversion(packet_collection_bits : np.ndarray , packet_collection : list[list]):
    #the list in the list[list] represents one packet
    random_packet_index= random.randrange(0, len(packet_collection))
    random_packet = packet_collection[random_packet_index]
    src_ip = BitArray(ipaddress.IPv4Address(random_packet[0]).packed)
    dst_ip = BitArray(ipaddress.IPv4Address(random_packet[1]).packed)
    sport = BitArray(format(random_packet[2], "016b"))
    manual_packet_bits = np.array(src_ip.tolist() + dst_ip.tolist() + sport.tolist(),dtype = np.bool)
    #TODO : is it easier to work with bitarrays or np.arrays
    assert np.array_equal(manual_packet_bits, packet_collection_bits), warnings.warn(f"packet {random_packet_index} is different from manual calculation ")
    return np.array_equal(manual_packet_bits, packet_collection_bits)


"""the current version only uses IPV4 packets and the first packets of a connection"""
def is_packet_usable(packet: Packet)-> bool:
    return is_first_packet(packet) and uses_IPV4(packet)

#to do 1 : work out if it is correct to only look at the packet at the beginning of a connection
#TODO : if the above is correct write a method that filters out any packet where the syn flag isnt set to one
def is_first_packet(packet : Packet) -> bool:
    pass

def uses_IPV4 (packet : Packet)-> bool:
    pass

"this class provides a set of normal packets to serve as the self set "

class Normal_Traffic:

        """This function converts the list of packages to a list of binaries"""
        def self_set_to_binary(self):
            pass

        def __init__(self):

            self.src_IP : list[str] = ["192.168.1.1", "192.168.1.2", "192.168.1.3", "192.168.1.4", "192.168.1.5"]
            self.dst_IP : list[str] = ["192.168.1.10", "192.168.1.20", "192.168.1.30", "192.168.1.40", "192.168.1.50"]
            self.sport : list[int] = [49152]
            #use this to test different values in the packet fields and using actual packets
            #self.self_set : list[Packet] = create_packet_set(PacketCollectionType.NORMAL,self.src_IP,self.dst_IP,self.sport)
            #use this to test if the packet list can be connected to the bit converter and the rest of the program
            self.dummy_self_set : list[list] = [["192.168.1.1","192.168.1.10", 49152],["192.168.1.2","192.168.1.20", 49152]]

        def __repr__(self) -> str:
            return f"{type(self).__name__}(self set = {self.dummy_self_set})"

class Mixed_Traffic:
    pass

self_set_object = Normal_Traffic()
first_collection= self_set_object.dummy_self_set
print("manually converted", "src_IP", BitArray(ipaddress.IPv4Address("192.168.1.1").packed),"dst_IP",BitArray(ipaddress.IPv4Address("192.168.1.10").packed), "sport", format(49152, "016b"))
convert_collection_to_bits(first_collection)
#"in scapy you access layers by class"
#src_value = firstpackage[IP].src
#print(type(src_value))
#print(src_value)
#repr(self_set_object)



manually converted src_IP bitarray('11000000101010000000000100000001') dst_IP bitarray('11000000101010000000000100001010') sport 1100000000000000
('src ip is ', bitarray('11000000101010000000000100000001'), 'in bits')
('dst ip is', bitarray('11000000101010000000000100001010'), 'in bits')
('sport ip is', bitarray('1100000000000000'), 'in bits')


/tmp/ipykernel_14777/2143648234.py:130: UserWarning: packet 0 is different from manual calculation 
  assert np.array_equal(manual_packet_bits, packet_collection_bits), warnings.warn(f"packet {random_packet_index} is different from manual calculation ")


AssertionError: None

In [5]:
import bitarray
from bitarray import bitarray as BitArray
import bitarray.util
import pytest

"AIS classes : antibody and lymphocyte are defined below"

class Antibody:
    #TODO put a __repr__ method here
    def __new__(cls, *args,**kwargs):
        print("Created a new instance of antibody.")
        return super().__new__(cls)

    def __init__(self):
        #TODO : how long is a packet with my fields and how to organise bits consistently
        self.detector : BitArray = bitarray.util.random_k(4,2,endian = "big")


    """this function returns true then one packet in the self set triggered one activation """
    #TODO : test if it returns true when input is same as detector
    def negative_selection(self, self_set):
        pass

    """this function takes packet collections and returns true if one packet activates a detector """
    def scan_collection(self, packet_collection) -> bool:
        #pick packets from collection in correct order
        #call r contiguous bits
        pass

    """performs r contigous bits comparison on each packet"""
    #TODO : test if it returns true when r contiguous bits match
    #TODO : test if it returns false when r contiguous bits dont match
    def scan_packet(self, packet : Packet) -> bool:
        #TODO : implement r contiguous bits
        #self.detector
        pass

class Lymphocyte:
    def __new__(cls, *args,**kwargs):
        print("Created a new instance of Lymphocyte.")
        #super allows access to parent class object we give its constructor
        #lymphocyte as an argument
        #we use a double underscore because line 7 calls a special method
        return super().__new__(cls)
    #self holds a reference to the current instance
    def __init__(self,antibody,stimulation):
        #TODO: change single antibody an antibody array
        self.antibody = Antibody()
        #TODO : implement stimulation
        self.stimulation = stimulation
    def __repr__(self) -> str:
        return f"{type(self).__name__}(antibody = {self.antibody.detector}, stimulation = {self.stimulation})"

    """activation threshold of lymphocyte is exceeded when lymphocyte detects X number of antigens in a short period of time"""
    def is_threshold_exceeded(self):
        pass

    """"If lymphocyte is not unique it is discarded"""
    def is_lymphocyte_unique(self):
        pass


lymphocyte = Lymphocyte("antifabody", "no stimulation")
lymphocyte.antibody.detects(lymphocyte.antibody.detector)
repr(lymphocyte)

Created a new instance of Lymphocyte.
Created a new instance of antibody.


"Lymphocyte(antibody = bitarray('0110'), stimulation = no stimulation)"